# Ant Mutilated – Comparative Evaluation

Evaluates and compares PPO and Hebbian Attractor Network (HAN) policies
across all morphological variants of the MuJoCo ant.

Accepts multiple training runs per algorithm (different seeds).
All runs are pooled to report SEM-based error bars and 95% CI.

**Only the *Configuration* cell needs to be edited.**

In [ ]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["MUJOCO_GL"] = "egl"

import json
from pathlib import Path
from typing import Callable, Dict, List, Optional

import jax
import jax.numpy as jnp
import numpy as np
import mujoco
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as scipy_stats
import scienceplots
from mujoco import mjx
from mujoco_playground._src.dm_control_suite import common
from omegaconf import OmegaConf
from orbax.checkpoint import StandardCheckpointer

# Brax / PPO
from brax.training import checkpoint as brax_checkpoint
from brax.training import networks as brax_networks
from brax.training.acme import running_statistics
from brax.training.agents.ppo import networks as ppo_networks

# Crawler / Kandel
import crawler_playground
from crawler_playground.envs.ant.ant import Run as AntRun, default_config
from crawler_playground.envs.ant.randomize import VALID_VARIANTS
from kandel.model import networks as kandel_networks
from kandel.utility.checkpoint_manager import (
    EPISODE_NAME,
    INDIVIDUAL_NAME,
    ENSEMBLE_NAME,
)
from kandel.utility.running_statistics import (
    init_state as kandel_init_state,
    normalize as kandel_normalize,
)

print("JAX devices:", jax.devices())

## Configuration

Set `ALGORITHM_RUNS` to point to your checkpoint directories.
Each entry must have keys `dir` (path) and `name` (label for plots).
The top-level keys (`"PPO"`, `"HAN (B)"`, etc.) are the algorithm names shown in the summary chart.

In [ ]:
# ── Fill in ────────────────────────────────────────────────────────────────────
ALGORITHM_RUNS: Dict[str, List[Dict]] = {
    "PPO": [
        # Each entry: one training run (checkpoint directory)
        {"dir": "/tmp/ant-mutilated-ppo-260319050306", "name": "1"},
        {"dir": "/tmp/ant-mutilated-ppo-260319043144", "name": "2"},
        {"dir": "/tmp/ant-mutilated-ppo-260319040017", "name": "3"},
        {"dir": "/tmp/ant-mutilated-ppo-260319032854", "name": "4"},
        {"dir": "/tmp/ant-mutilated-ppo-260319025729", "name": "5"},
        {"dir": "/tmp/ant-mutilated-ppo-260319022605", "name": "6"},
        {"dir": "/tmp/ant-mutilated-ppo-260319015440", "name": "7"},
        {"dir": "/tmp/ant-mutilated-ppo-260319012308", "name": "8"},
        {"dir": "/tmp/ant-mutilated-ppo-260319005141", "name": "9"},
        {"dir": "/tmp/ant-mutilated-ppo-260319002003", "name": "10"},
        {"dir": "/tmp/ant-mutilated-ppo-260318214550", "name": "11"},
        {"dir": "/tmp/ant-mutilated-ppo-260318211137", "name": "12"},
        {"dir": "/tmp/ant-mutilated-ppo-260318203733", "name": "13"},
        {"dir": "/tmp/ant-mutilated-ppo-260318200329", "name": "14"},
        {"dir": "/tmp/ant-mutilated-ppo-260318184137", "name": "15"},
    ],
    "HAN (B)": [
        {
            "dir": "/tmp/260318151246-ant-mutilated-mlp-hebbian-openai-es-42",
            "name": "1",
        },
        {
            "dir": "/tmp/260319104849-ant-mutilated-mlp-hebbian-openai-es-141",
            "name": "2",
        },
        {
            "dir": "/tmp/260319121309-ant-mutilated-mlp-hebbian-openai-es-142",
            "name": "3",
        },
        {
            "dir": "/tmp/260319133720-ant-mutilated-mlp-hebbian-openai-es-143",
            "name": "4",
        },
        {
            "dir": "/tmp/260319150131-ant-mutilated-mlp-hebbian-openai-es-144",
            "name": "5",
        },
        {
            "dir": "/tmp/260319162623-ant-mutilated-mlp-hebbian-openai-es-145",
            "name": "6",
        },
        {
            "dir": "/tmp/260319175128-ant-mutilated-mlp-hebbian-openai-es-146",
            "name": "7",
        },
        {
            "dir": "/tmp/260319191555-ant-mutilated-mlp-hebbian-openai-es-147",
            "name": "8",
        },
        {
            "dir": "/tmp/260319204009-ant-mutilated-mlp-hebbian-openai-es-148",
            "name": "9",
        },
        {
            "dir": "/tmp/260319220419-ant-mutilated-mlp-hebbian-openai-es-149",
            "name": "10",
        },
        {
            "dir": "/tmp/260319232831-ant-mutilated-mlp-hebbian-openai-es-150",
            "name": "11",
        },
        {
            "dir": "/tmp/260320232745-ant-mutilated-mlp-hebbian-openai-es-161",
            "name": "12",
        },
        {
            "dir": "/tmp/260321005050-ant-mutilated-mlp-hebbian-openai-es-162",
            "name": "13",
        },
        {
            "dir": "/tmp/260321021340-ant-mutilated-mlp-hebbian-openai-es-163",
            "name": "14",
        },
        {
            "dir": "/tmp/260321033635-ant-mutilated-mlp-hebbian-openai-es-164",
            "name": "15",
        },
    ],
    "HAN (E)": [
        {
            "dir": "/tmp/260320005953-ant-mutilated-mlp-hebbian-openai-es-141",
            "name": "1",
        },
        {
            "dir": "/tmp/260320022315-ant-mutilated-mlp-hebbian-openai-es-142",
            "name": "2",
        },
        {
            "dir": "/tmp/260320034623-ant-mutilated-mlp-hebbian-openai-es-143",
            "name": "3",
        },
        {
            "dir": "/tmp/260320063426-ant-mutilated-mlp-hebbian-openai-es-145",
            "name": "4",
        },
        {
            "dir": "/tmp/260320075734-ant-mutilated-mlp-hebbian-openai-es-146",
            "name": "5",
        },
        {
            "dir": "/tmp/260320092042-ant-mutilated-mlp-hebbian-openai-es-147",
            "name": "6",
        },
        {
            "dir": "/tmp/260320104347-ant-mutilated-mlp-hebbian-openai-es-148",
            "name": "7",
        },
        {
            "dir": "/tmp/260320120652-ant-mutilated-mlp-hebbian-openai-es-149",
            "name": "8",
        },
        {
            "dir": "/tmp/260320133000-ant-mutilated-mlp-hebbian-openai-es-150",
            "name": "9",
        },
        {
            "dir": "/tmp/260320145608-ant-mutilated-mlp-hebbian-openai-es-151",
            "name": "10",
        },
        {
            "dir": "/tmp/260320161926-ant-mutilated-mlp-hebbian-openai-es-152",
            "name": "11",
        },
        {
            "dir": "/tmp/260320174238-ant-mutilated-mlp-hebbian-openai-es-153",
            "name": "12",
        },
        {
            "dir": "/tmp/260320190551-ant-mutilated-mlp-hebbian-openai-es-154",
            "name": "13",
        },
        {
            "dir": "/tmp/260320202900-ant-mutilated-mlp-hebbian-openai-es-155",
            "name": "14",
        },
        {
            "dir": "/tmp/260320220042-ant-mutilated-mlp-hebbian-openai-es-156",
            "name": "15",
        },
    ],
}
# Rollouts per run per variant
NUM_SEEDS = 30
EPISODE_LENGTH = 500

# Which variants to evaluate (must be subset of VALID_VARIANTS)
EVAL_VARIANTS = ["none", "FR", "FL", "RL", "RR"]

# Variants seen during training vs unseen at test time
# Used only for chart shading — adjust to match the training configs used.
TRAIN_VARIANTS = ["none", "FR", "FL"]
TEST_VARIANTS = ["RL", "RR"]

# Kandel policy index (int → individual policy_N; None → ensemble mean)
KANDEL_POLICY_ID = 0
# ──────────────────────────────────────────────────────────────────────────────

## Shared environment factory

Both algorithms use the same `make_variant_env` to ensure a fair comparison.
For each variant the damaged MJX model is swapped in directly (no DR wrapper
is needed during evaluation).

In [ ]:
_ANT_XML_DIR = (
    Path(crawler_playground.__file__).parent / "envs" / "ant" / "xmls"
)


def make_variant_env(variant: str) -> AntRun:
    """Return an AntRun with the correct damaged XML for *variant*."""
    assert variant in VALID_VARIANTS, f"Unknown variant '{variant}'"
    env = AntRun(config=default_config())
    if variant != "none":
        xml_path = _ANT_XML_DIR / f"ant_mutilated_{variant}.xml"
        mj_model = mujoco.MjModel.from_xml_string(
            xml_path.read_text(), common.get_assets()
        )
        mj_model.opt.timestep = env.sim_dt
        env._mj_model = mj_model
        env._mjx_model = mjx.put_model(mj_model)
    return env

## Checkpoint loaders

In [ ]:
_KERNEL_INIT_KEYS = (
    "policy_network_kernel_init_fn",
    "value_network_kernel_init_fn",
    "q_network_kernel_init_fn",
    "mean_kernel_init_fn",
)


def load_ppo_run(checkpoint_dir: str):
    """Load a PPO checkpoint; return (inference_fn, step).

    Mirrors the loader in replay_ant_mutilated_ppo.ipynb exactly.
    """
    checkpoint_dir = Path(checkpoint_dir)
    step_dirs = sorted(
        [
            d
            for d in checkpoint_dir.iterdir()
            if d.is_dir() and d.name.isdigit()
        ],
        key=lambda d: int(d.name),
    )
    if not step_dirs:
        raise FileNotFoundError(f"No checkpoint dirs in {checkpoint_dir}")
    step = int(step_dirs[-1].name)
    ckpt_path = checkpoint_dir / f"{step:012d}"

    raw_params = brax_checkpoint.load(ckpt_path)
    normalizer_params, policy_params, _ = raw_params

    with open(ckpt_path / "ppo_network_config.json") as f:
        cfg_raw = json.load(f)

    kw = {k: v for k, v in cfg_raw["network_factory_kwargs"].items()}

    # Activation string → callable
    if "activation" in kw and isinstance(kw["activation"], str):
        kw["activation"] = brax_networks.ACTIVATION[kw["activation"]]

    # Kernel init strings → callables
    for key in _KERNEL_INIT_KEYS:
        if key in kw and kw[key] is not None:
            kw[key] = brax_networks.KERNEL_INITIALIZER[kw[key]]

    # observation_size may be serialised as {shape, dtype} dict
    obs_raw = cfg_raw["observation_size"]
    observation_size = (
        int(obs_raw["shape"][0]) if isinstance(obs_raw, dict) else int(obs_raw)
    )

    normalize_observations = cfg_raw.get("normalize_observations", False)
    preprocess_fn = (
        running_statistics.normalize
        if normalize_observations
        else (lambda x, y: x)
    )

    ppo_network = ppo_networks.make_ppo_networks(
        observation_size=observation_size,
        action_size=cfg_raw["action_size"],
        preprocess_observations_fn=preprocess_fn,
        **kw,
    )
    make_policy_ = ppo_networks.make_inference_fn(ppo_network)
    params = (normalizer_params, policy_params)
    inference_fn = jax.jit(make_policy_(params, deterministic=True))

    return inference_fn, step


print("PPO loader ready.")

In [ ]:
def load_kandel_run(checkpoint_dir: str, policy_id: Optional[int] = 0):
    """Load a Kandel/HAN checkpoint; return (policy, params_init, norm_state, cfg, step).

    norm_state has count=0 — frozen checkpoint statistics, Welford updates disabled.
    Mirrors the loader in replay_ant_mutilated_kandel.ipynb exactly.
    """
    checkpoint_dir = Path(checkpoint_dir)
    cfg = OmegaConf.load(checkpoint_dir / "metadata.yaml")

    episode_dirs = sorted(
        [
            d
            for d in checkpoint_dir.iterdir()
            if d.is_dir() and d.name.startswith(EPISODE_NAME + "_")
        ],
        key=lambda d: int(d.name.split("_")[-1]),
    )
    if not episode_dirs:
        raise FileNotFoundError(f"No episode dirs in {checkpoint_dir}")
    step = int(episode_dirs[-1].name.split("_")[-1])

    meta_path = (
        checkpoint_dir / f"{EPISODE_NAME}_{step}" / "checkpoint_metadata.json"
    )
    with open(meta_path) as f:
        meta = json.load(f)

    norm_data = meta["norm_state"]
    dummy_obs = jnp.zeros((cfg.observation_dim,))
    norm_state = kandel_init_state(dummy_obs).replace(
        mean=jnp.array(norm_data["mean"]),
        std=jnp.array(norm_data["std"]),
        summed_variance=jnp.array(norm_data["summed_variance"]),
        count=0,
    )

    policy = kandel_networks[cfg.policy_id](
        input_dim=cfg.observation_dim,
        output_dim=cfg.action_dim,
        **cfg.policy,
    )
    dummy_params = policy.initialize(jax.random.key(0))

    policy_subdir = (
        f"{INDIVIDUAL_NAME}_{policy_id}"
        if isinstance(policy_id, int)
        else ENSEMBLE_NAME
    )
    policy_path = checkpoint_dir / f"{EPISODE_NAME}_{step}" / policy_subdir
    policy_params_init = StandardCheckpointer().restore(
        policy_path, dummy_params
    )

    return policy, policy_params_init, norm_state, cfg, step


print("Kandel loader ready.")

## Batched rollout helpers

Each helper returns a `jax.jit(jax.vmap(single_rollout))` function.
Calling it with an integer array of shape `[NUM_SEEDS]` returns per-seed
total rewards of shape `[NUM_SEEDS]`.

In [ ]:
def make_batched_rollout_ppo(env: AntRun, inference_fn: Callable):
    """vmap+scan rollout for a stateless PPO inference_fn.

    Mirrors make_batched_rollout in replay_ant_mutilated_ppo.ipynb exactly.
    """
    _reset = env.reset
    _step = env.step

    def single(seed: int):
        rng = jax.random.key(seed)
        rng, rng_reset = jax.random.split(rng)
        state = _reset(rng_reset)

        def scan_step(carry, _):
            state, rng, total, active = carry
            rng, rng_act = jax.random.split(rng)
            actions, _ = inference_fn(state.obs, rng_act)
            new_state = _step(state, actions)
            active = active * (1.0 - new_state.done)
            total = total + new_state.reward * active
            return (new_state, rng, total, active), None

        (_, _, total_reward, _), _ = jax.lax.scan(
            scan_step,
            (state, rng, jnp.zeros(()), jnp.ones(())),
            None,
            length=EPISODE_LENGTH,
        )
        return total_reward

    return jax.jit(jax.vmap(single))

In [ ]:
def make_batched_rollout_kandel(
    env: AntRun,
    policy,
    policy_params_init,
    norm_state,
    cfg,
):
    """vmap+scan rollout for a stateful Hebbian policy.

    Hebbian weights are carried through the scan and updated every
    `relative_update_steps` steps via lax.cond, matching the training loop.
    The frozen norm_state (count=0) is used for observation normalisation.
    Rewards are masked after done=True, matching the training loop's
    track_terminations=True behaviour.
    """
    env_reset_j = jax.jit(env.reset)
    env_step_j = jax.jit(env.step)
    obs_normalize_fn = jax.jit(kandel_normalize)
    policy_forward_fn = jax.jit(policy.forward)
    policy_update_fn = jax.jit(policy.update)
    policy_reset_fn = jax.jit(policy.reset)

    sim_interval = cfg.simulation_interval
    update_interval = cfg.update_interval
    rollout_length = cfg.rollout_length
    action_clip_min = cfg.action_clip_min
    action_clip_max = cfg.action_clip_max

    total_time = rollout_length * sim_interval
    total_steps = int(total_time / min(sim_interval, update_interval))
    relative_update_steps = int(
        total_steps / int(total_time / update_interval)
    )

    def single(seed: int):
        key = jax.random.key(seed)
        rng_env, rng_policy = jax.random.split(key)
        pp = policy_reset_fn(policy_params_init, rng_policy)
        es = env_reset_j(rng_env)

        def scan_fn(carry, t):
            pp, es, active = carry
            obs = obs_normalize_fn(es.obs, norm_state)
            pp, actions = policy_forward_fn(pp, obs)
            actions = jnp.clip(actions, action_clip_min, action_clip_max)
            es = env_step_j(es, actions)
            active = active * (1 - es.done.astype(jnp.int32))
            reward = es.reward * active.astype(jnp.float32)
            pp = jax.lax.cond(
                t % relative_update_steps == 0,
                policy_update_fn,
                lambda p: p,
                pp,
            )
            return (pp, es, active), reward

        _, rewards = jax.lax.scan(
            scan_fn,
            (pp, es, jnp.ones((), dtype=jnp.int32)),
            jnp.arange(rollout_length),
        )
        return jnp.sum(rewards)

    return jax.jit(jax.vmap(single))

## Evaluation loop

Iterates over all algorithms and runs, producing `pooled[algo][variant]`
(a flat list of all seeds × runs rewards) and `per_run[algo][run_name][variant]`
(per-run means for the dot overlay).

In [ ]:
pooled: Dict[str, Dict[str, np.ndarray]] = {}
per_run: Dict[str, Dict[str, Dict[str, float]]] = {}

seed_batch = jnp.arange(NUM_SEEDS)

for algo, runs in ALGORITHM_RUNS.items():
    if not runs:
        print(f"[{algo}] No runs configured — skipping.")
        continue

    pooled[algo] = {v: [] for v in EVAL_VARIANTS}
    per_run[algo] = {}

    for run in runs:
        run_dir = run["dir"]
        run_name = run["name"]
        print(f"\n{'─' * 55}")
        print(f"[{algo}]  {run_name}  ({run_dir})")

        run_variant_rewards = {}

        if algo == "PPO":
            infer_fn, step = load_ppo_run(run_dir)
            print(f"  checkpoint step: {step:,}")
            for variant in EVAL_VARIANTS:
                env = make_variant_env(variant)
                batch_fn = make_batched_rollout_ppo(env, infer_fn)
                rewards = np.array(batch_fn(seed_batch))
                run_variant_rewards[variant] = rewards
                pooled[algo][variant].append(rewards)
                split = "train" if variant in TRAIN_VARIANTS else "test "
                print(
                    f"  [{split}] {variant:4s}  {rewards.mean():7.1f} ± {rewards.std(ddof=1) / np.sqrt(NUM_SEEDS):.1f} SEM"
                )

        else:  # Kandel / HAN
            policy, policy_params_init, norm_state, cfg, step = (
                load_kandel_run(run_dir, policy_id=KANDEL_POLICY_ID)
            )
            print(f"  checkpoint step: {step}")
            for variant in EVAL_VARIANTS:
                env = make_variant_env(variant)
                batch_fn = make_batched_rollout_kandel(
                    env, policy, policy_params_init, norm_state, cfg
                )
                rewards = np.array(batch_fn(seed_batch))
                run_variant_rewards[variant] = rewards
                pooled[algo][variant].append(rewards)
                split = "train" if variant in TRAIN_VARIANTS else "test "
                print(
                    f"  [{split}] {variant:4s}  {rewards.mean():7.1f} ± {rewards.std(ddof=1) / np.sqrt(NUM_SEEDS):.1f} SEM"
                )

        per_run[algo][run_name] = {
            v: float(np.mean(run_variant_rewards[v])) for v in EVAL_VARIANTS
        }

# Concatenate per-algo per-variant lists into arrays
for algo in pooled:
    for v in EVAL_VARIANTS:
        if pooled[algo][v]:
            pooled[algo][v] = np.concatenate(pooled[algo][v])

print("\nEvaluation complete.")

## Summary statistics table

In [ ]:
rows = []
for algo in pooled:
    for variant in EVAL_VARIANTS:
        rewards = pooled[algo][variant]
        if not isinstance(rewards, np.ndarray) or len(rewards) == 0:
            continue
        n = len(rewards)
        mu = float(np.mean(rewards))
        sem = float(np.std(rewards, ddof=1) / np.sqrt(n))
        t = scipy_stats.t.ppf(0.975, df=n - 1)
        ci = t * sem
        split = "Train" if variant in TRAIN_VARIANTS else "Test"
        rows.append(
            {
                "Algorithm": algo,
                "Variant": variant,
                "Split": split,
                "N": n,
                "Mean": round(mu, 1),
                "SEM": round(sem, 1),
                "95% CI": f"{mu:.1f} ± {ci:.1f}",
            }
        )

df = pd.DataFrame(rows)
if not df.empty:
    display(df.set_index(["Algorithm", "Variant"]))
else:
    print("No data yet — run the evaluation cell first.")

## Summary chart

One group of bars per variant; one bar per algorithm.
Error bars show ±SEM over all pooled seeds × runs.
Per-run means are overlaid as small dots.

In [ ]:
plt.style.use(["science", "ieee"])

# EPFL-inspired palette
epfl_red = "#B51F1F"
epfl_dark = "#413D3A"
epfl_teal = "#007480"
epfl_grey = "#888888"

train_bg_color = "#D0661C"
test_bg_color = "#46b361"

_default_colors = [epfl_red, epfl_dark, epfl_teal, epfl_grey]
algorithms = list(pooled.keys())
algo_colors = {
    a: _default_colors[i % len(_default_colors)]
    for i, a in enumerate(algorithms)
}

variants_display = EVAL_VARIANTS
n_variants = len(variants_display)
n_methods = len(algorithms)
width = 0.7 / max(n_methods, 1)
x = np.arange(n_variants)

fig, ax = plt.subplots(figsize=(3, 1.75))

for i, algo in enumerate(algorithms):
    means, cis = [], []
    for variant in variants_display:
        if algo in per_run and per_run[algo]:
            run_means = np.array(
                [
                    per_run[algo][r][variant]
                    for r in per_run[algo]
                    if variant in per_run[algo][r]
                ]
            )
            n = len(run_means)
            mu = float(run_means.mean())
            t = scipy_stats.t.ppf(0.975, df=n - 1) if n > 1 else 0.0
            ci = (
                t * float(run_means.std(ddof=1) / np.sqrt(n)) if n > 1 else 0.0
            )
        else:
            mu, ci = 0.0, 0.0
        means.append(mu)
        cis.append(ci)

    n_runs = len(ALGORITHM_RUNS[algo])
    offsets = x + (i - (n_methods - 1) / 2) * width
    ax.bar(
        offsets,
        means,
        width,
        label=f"{algo}",
        color=algo_colors[algo],
        yerr=cis,
        capsize=2,
        ecolor="#333333",
        error_kw={"elinewidth": 1.0, "capthick": 1.0},
    )

    # Per-run means as dots
    # if algo in per_run:
    #     for run_name, variant_means in per_run[algo].items():
    #         dot_y = [variant_means.get(v, np.nan) for v in variants_display]
    #         ax.scatter(offsets, dot_y, s=3, color="#000000", zorder=5, linewidths=0, alpha=0.4)

# Train / test region shading
group_left = x - (n_methods / 2) * width - width * 0.1
group_right = x + (n_methods / 2) * width + width * 0.1

train_indices = [
    i for i, v in enumerate(variants_display) if v in TRAIN_VARIANTS
]
test_indices = [
    i for i, v in enumerate(variants_display) if v in TEST_VARIANTS
]

xlim = ax.get_xlim()
boundary = (group_right[train_indices[-1]] + group_left[test_indices[0]]) / 2
if train_indices:
    ax.axvspan(
        xlim[0],
        boundary,
        facecolor=train_bg_color,
        alpha=0.15,
        zorder=0,
        edgecolor="none",
    )
if test_indices:
    ax.axvspan(
        boundary,
        xlim[1],
        facecolor=test_bg_color,
        alpha=0.15,
        zorder=0,
        edgecolor="none",
    )

if train_indices and test_indices:
    ax.axvline(boundary, color="#000000", linewidth=0.75, zorder=1)
    ax.set_xlim(group_left[0] - 0.15, group_right[-1] + 0.15)
    xlim = ax.get_xlim()
    span = xlim[1] - xlim[0]
    tc = (group_left[train_indices[0]] + group_right[train_indices[-1]]) / 2
    tsc = (group_left[test_indices[0]] + group_right[test_indices[-1]]) / 2
    ax.text(
        (tc - xlim[0]) / span,
        1.02,
        "Train",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        color=train_bg_color,
    )
    ax.text(
        (tsc - xlim[0]) / span,
        1.02,
        "Test (unseen)",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        color=test_bg_color,
    )

ax.set_xticks(x)
ax.set_xticklabels([v if v != "none" else "None" for v in variants_display])
ax.set_xlabel("Mutilated Leg")
ax.set_ylabel("Acc. Return (-)")
ax.minorticks_off()
ax.grid(axis="y", alpha=0.3, zorder=0)
ax.set_axisbelow(True)
ax.legend(frameon=True, loc="upper right", fontsize=6)

fig.tight_layout()
fig.savefig("ant_mutilated_results.pdf", dpi=300)
plt.show()